# AIC211 · Topic 2 · TV3 (Trí, SE211087) — D1+: chuẩn hóa dấu “/” cuối

**Ý tưởng:** E2b cho thấy RF học luật “có / cuối = sạch”. D1+ = D1 cũ **cộng** bỏ dấu “/” khi path chỉ là “/” và không có query. Áp dụng ở **cả lúc train lẫn lúc test**, nên model không còn nhìn thấy dấu “/” đó nữa.

**Bốn cấu hình train trong notebook này** (cùng tham số RF với `RF_F1.joblib`, ngưỡng FPR 1% trên val A):

| Tên | Feature | Tiền xử lý | D3 | So với cấu hình chính thức |
|---|---|---|---|---|
| `D1p` | F1 | D1+ | không | `D1` |
| `D1p+D2` | F1 + D2 | D1+ | không | `D1+D2` |
| `D1+D2+D3rt` | F1 + D2 | D1 | có | đối chứng: train lại D1+D2+D3 theo đúng quy trình D3 ở đây |
| `D1p+D2+D3` | F1 + D2 | D1+ | có | `D1+D2+D3rt` |

Không sửa file cũ nào. Chỉ tạo thêm: `src/defenses_plus.py`, `models/RF_*_D1p*.joblib`, `models/RF_F1D2_D1_D3rt.joblib`, `models/thresholds_D1p.json`, `results/E4_D1p_summary.csv`, và (khi `LOG = True`) các dòng E4, E4_clean, E2b_slash mới trong `results.csv`.

| Ô | Việc | Thời gian ước tính (Colab) |
|---|---|---|
| 1–3 | Nạp, ghi `defenses_plus.py`, kiểm thử | 3 phút |
| 4 | Train 4 cấu hình | 5–15 phút |
| 5 | Đánh giá 8 cấu hình: sạch, dấu “/”, tấn công 3 seed | 10–20 phút |
| 6 | Bảng so sánh + ghi kết quả | vài giây |

In [ ]:
# Ô 1 — Cài thư viện + nối Google Drive
import os, sys, importlib
from google.colab import drive
drive.mount('/content/drive')
!pip -q install tldextract rapidfuzz shap xgboost
ROOT = '/content/drive/MyDrive/AIC211_URL'
sys.path.insert(0, f'{ROOT}/src')
print('ROOT =', ROOT)

In [ ]:
# Ô 2 — Ghi src/defenses_plus.py (file MỚI, không sửa defenses.py)
CODE = r"""
from urllib.parse import urlsplit, urlunsplit
from defenses import d1_normalize

def d1p_normalize(url):
    # D1+ = D1 + bỏ dấu '/' khi path chỉ là '/' và không có query (trang chủ luôn về một dạng)
    u = d1_normalize(url)
    try:
        p = urlsplit(u if '://' in u else 'http://' + u)
    except ValueError:
        return u
    if p.path == '/' and not p.query:
        return urlunsplit((p.scheme, p.netloc, '', '', ''))
    return u
"""
open(f'{ROOT}/src/defenses_plus.py', 'w', encoding='utf-8').write(CODE)
import defenses_plus; importlib.reload(defenses_plus)
from defenses_plus import d1p_normalize
for u in ['http://suttontrust.com', 'http://suttontrust.com/', 'http://example.tk/?utm_medium=email',
          'http://a.com/login?id=7&utm_source=mail', 'http://a.com/x/']:
    print(f'{u:45s} -> {d1p_normalize(u)}')

In [ ]:
# Ô 3 — Nạp code, dữ liệu, model gốc, 300 URL độc (giống tuần 6, 7)
import common; importlib.reload(common)
from common import *
import json, joblib, random, time
import numpy as np, pandas as pd
from sklearn.base import clone
import attacks, e3, defenses, d2, e4
for m in (attacks, e3, defenses, d2, e4): importlib.reload(m)
from attacks import to_attack_form, FREE_OPS, PAID_OPS
from e3 import make_score_fn, attack_success_rate, random_attack
from defenses import d1_normalize
from d2 import make_F1D2
from e4 import load_configs

A = pd.read_parquet(f'{ROOT}/data/processed/A.parquet')
B = pd.read_parquet(f'{ROOT}/data/processed/B.parquet')
tr, va, te = (A.split == 'train').values, (A.split == 'val').values, (A.split == 'test').values
benign_domains = set(A[A.label == 0]['reg_domain'])
ALL_OPS = FREE_OPS + PAID_OPS
OPS_D3 = [o for o in ALL_OPS if o != 'T6']        # D3 giữ lại T6 để kiểm tra tổng quát

N = 300
rng0 = random.Random(0)
MAL = rng0.sample([to_attack_form(u) for u in A[te & (A.label == 1).values]['url_norm']], N)
rf_base = joblib.load(f'{ROOT}/models/RF_F1.joblib')
FEAT = {'F1': make_F1, 'F1D2': make_F1D2}

def pre_frame(urls_norm, labels, pre):
    # URL -> tiền xử lý (D1 hoặc D1+) -> chuẩn hóa -> khung có url_norm, host, rest
    u = [normalize_url(pre('http://' + x)) for x in urls_norm]
    out = pd.DataFrame({'url_norm': u, 'label': np.asarray(labels)})
    parts = out['url_norm'].map(split_parts)
    out['host'] = parts.str[0]; out['rest'] = parts.str[1]
    return out
print('A:', A.shape, '| B:', B.shape, '| số URL tấn công:', len(MAL), '| URL đầu:', attacks.defang(MAL[0]))

In [ ]:
# Ô 4 — Train 4 cấu hình (cùng tham số RF với RF_F1), ngưỡng FPR 1% trên val A. Train xong lưu file rồi giải phóng RAM.
import gc
def d3_augment(pre, seed=0, frac=0.5):
    # D3: lấy 50% URL độc của train A, sửa ngẫu nhiên 1–3 biến đổi (không dùng T6), thêm vào train
    rng = random.Random(seed)
    mal = [to_attack_form(u) for u in A[tr & (A.label == 1).values]['url_norm']]
    pick = rng.sample(mal, int(frac * len(mal)))
    adv = [random_attack(u, rng.choice([1, 2, 3]), OPS_D3, rng, benign_domains)[0] for u in pick]
    return pre_frame([normalize_url(x) for x in adv], np.ones(len(adv), int), pre)

SPECS = {'D1p':         ('RF_F1_D1p',         'F1',   d1p_normalize, False),
         'D1p+D2':      ('RF_F1D2_D1p',       'F1D2', d1p_normalize, False),
         'D1+D2+D3rt':  ('RF_F1D2_D1_D3rt',   'F1D2', d1_normalize,  True),
         'D1p+D2+D3':   ('RF_F1D2_D1p_D3',    'F1D2', d1p_normalize, True)}
THR_NEW = {}
MODEL_DIR = '/content/d1p_models'
os.makedirs(MODEL_DIR, exist_ok=True)
for name, (mfile, fs, pre, use_d3) in SPECS.items():
    t0 = time.time()
    fa = pre_frame(A['url_norm'], A['label'], pre)
    Xtr, ytr = FEAT[fs](fa[tr]), fa.label[tr]
    if use_d3:
        aug = d3_augment(pre)
        Xtr = pd.concat([Xtr, FEAT[fs](aug)], ignore_index=True); ytr = pd.concat([ytr, aug.label], ignore_index=True)
    m = clone(rf_base).fit(Xtr, ytr)
    THR_NEW[name] = threshold_at_fpr(fa.label[va], get_scores(m, FEAT[fs](fa[va])))
    joblib.dump(m, f'{MODEL_DIR}/{mfile}.joblib')
    print(f'{name:12s} train {len(ytr):,} URL | ngưỡng {THR_NEW[name]:.4f} | {time.time() - t0:.0f}s')
    del m, Xtr, ytr, fa; gc.collect()
with open(f'{ROOT}/models/thresholds_D1p.json', 'w') as f:
    json.dump(THR_NEW, f, indent=2)

In [ ]:
# Ô 5 — Đánh giá từng cấu hình (sạch A-test/B, thử dấu '/', tấn công seed 0,1,2). Nạp từng model một để đỡ tốn RAM.
OFFICIAL = {c[0]: c for c in e4.CONFIGS if c[0] in ['none', 'D1', 'D1+D2', 'D1+D2+D3']}
def load_cfg(name):
    if name in OFFICIAL:
        _, mfile, tfile, tkey, fs, use_d1 = OFFICIAL[name]
        thr = json.load(open(f'{ROOT}/models/{tfile}'))[tkey]; pre = d1_normalize if use_d1 else None
    else:
        mfile, fs, pre, _ = SPECS[name]; thr = THR_NEW[name]
    m = joblib.load(f'{ROOT}/models/{mfile}.joblib' if name in OFFICIAL else f'{MODEL_DIR}/{mfile}.joblib')
    return m, {'score': make_score_fn(m, FEAT[fs], pre), 'thr': thr}

ORDER = ['none', 'D1', 'D1p', 'D1+D2', 'D1p+D2', 'D1+D2+D3', 'D1+D2+D3rt', 'D1p+D2+D3']
bh = B[(B.rest.fillna('').str.len() <= 1) & (B.label == 0)].sample(3000, random_state=0)
orig_b = ['http://' + u for u in bh['url_norm']]; mod_b = ['http://' + u.rstrip('/') + '/' for u in bh['url_norm']]
CLEAN, SL, ATT = [], [], []
t0 = time.time()
for name in ORDER:
    m, cfg = load_cfg(name)
    for src, df in [('A-test', A[te]), ('B', B)]:
        met = evaluate(df['label'], np.asarray(cfg['score'](['http://' + u for u in df['url_norm']])), cfg['thr'])
        CLEAN.append({'config': name, 'source': src, **{k: met[k] for k in ['f1', 'recall', 'fpr', 'pr_auc']}})
    SL.append({'config': name, 'blocked_before': float((np.asarray(cfg['score'](orig_b)) > cfg['thr']).mean()),
               'blocked_after': float((np.asarray(cfg['score'](mod_b)) > cfg['thr']).mean())})
    for seed in [0, 1, 2]:
        for mode, k, ops, opsname in [('greedy', 3, ALL_OPS, 'all'), ('random', 3, ALL_OPS, 'all'), ('greedy', 1, ['T6'], 'T6')]:
            r, _ = attack_success_rate(MAL, cfg['score'], cfg['thr'], k, ops, mode, benign_domains, seed=seed)
            ATT.append({'config': name, 'seed': seed, 'mode': mode, 'ops': opsname, 'k': k, **r})
    del m, cfg; gc.collect()
    print(f'{name:12s} xong ({(time.time() - t0) / 60:.1f} phút)')
CLEAN, SL, ATT = pd.DataFrame(CLEAN), pd.DataFrame(SL), pd.DataFrame(ATT)

In [ ]:
# Ô 6 — Bảng so sánh (mean ± std qua 3 seed) + ghi kết quả khi LOG = True (báo nhóm trước)
LOG = False
g = ATT.groupby(['config', 'mode', 'ops'])[['asr', 'recall_attack', 'mean_queries']].agg(['mean', 'std'])
ca, cb = CLEAN[CLEAN.source == 'A-test'].set_index('config'), CLEAN[CLEAN.source == 'B'].set_index('config')
tab = pd.DataFrame({
    'F1 sạch A': ca['f1'], 'recall A': ca['recall'], 'FPR A': ca['fpr'], 'FPR B': cb['fpr'], 'recall B': cb['recall'], 'PR-AUC B': cb['pr_auc'],
    'chặn nhầm trang chủ B (thêm /)': SL.set_index('config')['blocked_after'],
    'ASR tham lam': g.xs(('greedy', 'all'), level=['mode', 'ops'])[('asr', 'mean')],
    'ASR ngẫu nhiên': g.xs(('random', 'all'), level=['mode', 'ops'])[('asr', 'mean')],
    'ASR ngẫu nhiên std': g.xs(('random', 'all'), level=['mode', 'ops'])[('asr', 'std')],
    'ASR chỉ T6': g.xs(('greedy', 'T6'), level=['mode', 'ops'])[('asr', 'mean')],
    'recall dưới tham lam': g.xs(('greedy', 'all'), level=['mode', 'ops'])[('recall_attack', 'mean')],
    'truy vấn TB': g.xs(('greedy', 'all'), level=['mode', 'ops'])[('mean_queries', 'mean')],
}).loc[ORDER]
tab.round(3).to_csv(f'{ROOT}/results/E4_D1p_summary.csv')
pd.set_option('display.width', 250)
print(tab.round(3).to_string())

if LOG:
    for name in SPECS:
        for r in CLEAN[CLEAN.config == name].itertuples():
            log_result(ROOT, 'E4_clean', 'RF', r.source, {'f1': r.f1, 'recall': r.recall, 'fpr': r.fpr, 'pr_auc': r.pr_auc}, defense=name)
        for r in ATT[ATT.config == name].itertuples():
            log_result(ROOT, 'E4', 'RF', 'A-test', {'asr': r.asr, 'recall_clean': r.recall_clean, 'recall_attack': r.recall_attack,
                       'n_detected': r.n_detected, 'mean_queries': r.mean_queries}, defense=name, seed=r.seed, k=r.k, ops=r.ops, mode=r.mode)
        row = SL[SL.config == name].iloc[0]
        log_result(ROOT, 'E2b_slash', name, 'B sạch, trang chủ (không "/")',
                   {'blocked_before': row.blocked_before, 'blocked_after': row.blocked_after}, ops='add_slash')
    print('Đã ghi kết quả của', list(SPECS), 'vào results.csv')
else:
    print('Chưa ghi results.csv (LOG = False). Đã lưu results/E4_D1p_summary.csv.')

## Cách đọc
- **FPR B:** nếu D1+ làm FPR trên B giảm mạnh từ 100%, D1+ sửa được lỗi đổi nguồn – kết quả quan trọng cho H1, H3.
- **ASR tham lam:** so `D1p` với `D1`, `D1p+D2+D3` với `D1+D2+D3rt`. Giảm rõ = D1+ chặn được đường lọt bằng dấu “/”.
- **F1 sạch A, FPR A:** cái giá của D1+ trên dữ liệu sạch.
- Nếu không cải thiện thì vẫn là kết quả âm có giá trị: model còn những “mẹo” khác ngoài dấu “/”.